# Trabajo Práctico Grupal: Programación Orientada a Objetos
## "Biblioteca POO"

**Duración:** 45 minutos de trabajo en grupo + 15 minutos de reflexión individual
**Modalidad:** grupos de 3 o 4 integrantes (la reflexión final es **individual y en papel**)
**Entrega:** repositorio público en GitHub (el link se entrega al final de la clase)

---

### ¡Atención! Esta vez no hay código de base
En el TP anterior las clases venían armadas con `# TODO`. **Ahora las escriben ustedes desde cero**: la clase, el `__init__`, los atributos, los métodos y las validaciones.
Lo único fijo son los **nombres** de las clases y los métodos (las pruebas automáticas los usan), así que respétenlos tal cual.

### Objetivos
Al terminar la clase, el grupo va a haber:
1. Diseñado y escrito **3 clases** sin plantilla.
2. Usado **encapsulamiento** (atributos protegidos, `@property` y validaciones).
3. Hecho que los objetos **se relacionen** entre sí (una Biblioteca que tiene Libros y Socios, y Socios que tienen Libros).
4. Subido el trabajo a **GitHub** con al menos 3 commits.

---
## Parte 1 — Clase `Libro` (8 min)

Un libro tiene **título**, **autor**, **ISBN** y sabe si está **disponible** o **prestado**. Cuando se crea, está disponible.

Tiene que permitir:
- Consultar si está disponible con `libro.disponible` (es una **propiedad**: se lee, pero **no se puede asignar** desde afuera).
- `prestar()`: lo marca como prestado. Si ya estaba prestado, lanza `ValueError`.
- `devolver()`: lo marca como disponible. Si no estaba prestado, lanza `ValueError`.
- Al hacer `print(libro)` mostrar algo como: `[978-1] Rayuela - Cortázar (Disponible)` o `(Prestado)`.

El constructor se usa así: `Libro(titulo, autor, isbn)`

In [1]:
class Libro:
    """Un libro de la biblioteca: sabe si está disponible o prestado."""

    def __init__(self, titulo, autor, isbn):
        self._titulo = titulo
        self._autor = autor
        self._isbn = isbn
        self._disponible = True

    @property
    def titulo(self):
        return self._titulo

    @property
    def autor(self):
        return self._autor

    @property
    def isbn(self):
        return self._isbn

    @property
    def disponible(self):
        return self._disponible

    def prestar(self):
        if not self._disponible:
            raise ValueError(f"El libro '{self._titulo}' ya está prestado")
        self._disponible = False

    def devolver(self):
        if self._disponible:
            raise ValueError(f"El libro '{self._titulo}' no está prestado")
        self._disponible = True

    def __str__(self):
        estado = "Disponible" if self._disponible else "Prestado"
        return f"[{self._isbn}] {self._titulo} - {self._autor} ({estado})"

In [2]:
# Prueben su clase acá
libro = Libro("Rayuela", "Cortázar", "978-1")
print(libro)            # [978-1] Rayuela - Cortázar (Disponible)
libro.prestar()
print(libro.disponible) # False

[978-1] Rayuela - Cortázar (Disponible)
False


---
## Parte 2 — Clase `Socio` (8 min)

Un socio tiene **nombre**, **DNI** y una **lista de libros** que tiene prestados (arranca vacía).
**Un socio puede tener como máximo 3 libros a la vez.** Ese 3 no tiene que estar "suelto" en el medio del código: piensen dónde conviene guardarlo.

Tiene que permitir:
- `libros`: **propiedad** que devuelve los libros que tiene el socio.
- `puede_pedir()`: devuelve `True` si todavía no llegó al máximo.
- `agregar_libro(libro)`: suma el libro a su lista. Si ya tiene el máximo, lanza `ValueError`.
- `quitar_libro(libro)`: lo saca de su lista. Si el socio no tiene ese libro, lanza `ValueError`.
- Al hacer `print(socio)` mostrar algo como: `Ana (DNI 30111222) - 2 libro(s) prestado(s)`.

El constructor se usa así: `Socio(nombre, dni)`

> Ojo: `Socio` **no** cambia el estado del libro (no llama a `prestar()` ni a `devolver()`). Eso lo coordina la `Biblioteca`.

In [3]:
class Socio:
    """Un socio de la biblioteca, con la lista de libros que tiene prestados."""

    MAX_LIBROS = 3

    def __init__(self, nombre, dni):
        self._nombre = nombre
        self._dni = dni
        self._libros = []

    @property
    def nombre(self):
        return self._nombre

    @property
    def dni(self):
        return self._dni

    @property
    def libros(self):
        # Devolvemos una copia para que no se pueda modificar la lista desde afuera
        return list(self._libros)

    def puede_pedir(self):
        return len(self._libros) < self.MAX_LIBROS

    def agregar_libro(self, libro):
        if not self.puede_pedir():
            raise ValueError(f"{self._nombre} ya tiene el máximo de {self.MAX_LIBROS} libros")
        self._libros.append(libro)

    def quitar_libro(self, libro):
        if libro not in self._libros:
            raise ValueError(f"{self._nombre} no tiene ese libro")
        self._libros.remove(libro)

    def __str__(self):
        return f"{self._nombre} (DNI {self._dni}) - {len(self._libros)} libro(s) prestado(s)"

In [4]:
# Prueben su clase acá
ana = Socio("Ana", "30111222")
ana.agregar_libro(Libro("Ficciones", "Borges", "978-2"))
print(ana)              # Ana (DNI 30111222) - 1 libro(s) prestado(s)
print(ana.puede_pedir())  # True

Ana (DNI 30111222) - 1 libro(s) prestado(s)
True


---
## Parte 3 — Clase `Biblioteca`: la que coordina todo (12 min)

Una biblioteca tiene un **nombre**, un conjunto de **libros** y un conjunto de **socios**. Piensen qué estructura conviene usar para encontrarlos rápido (por ISBN y por DNI).

Tiene que permitir:
| Método | Qué hace |
|---|---|
| `agregar_libro(libro)` | Agrega el libro. Si ya hay uno con ese ISBN, lanza `ValueError` |
| `registrar_socio(socio)` | Registra al socio. Si ya hay uno con ese DNI, lanza `ValueError` |
| `prestar(isbn, dni)` | Presta el libro al socio. Lanza `ValueError` si el libro o el socio no existen, si el libro no está disponible o si el socio llegó al máximo |
| `devolver(isbn, dni)` | El socio devuelve el libro. Lanza `ValueError` si no existen o si ese socio no tiene ese libro |
| `libros_disponibles()` | Devuelve una **lista** con los libros disponibles |

El constructor se usa así: `Biblioteca(nombre)`

> **Importante:** si un préstamo falla, **nada** tiene que quedar a medio hacer. Por ejemplo, si el socio ya tiene 3 libros, el libro tiene que seguir disponible después del error.

In [5]:
class Biblioteca:
    """Coordina los préstamos entre los libros y los socios."""

    def __init__(self, nombre):
        self._nombre = nombre
        # Diccionarios para encontrar rápido por ISBN y por DNI
        self._libros = {}
        self._socios = {}

    @property
    def nombre(self):
        return self._nombre

    def agregar_libro(self, libro):
        if libro.isbn in self._libros:
            raise ValueError(f"Ya existe un libro con ISBN {libro.isbn}")
        self._libros[libro.isbn] = libro

    def registrar_socio(self, socio):
        if socio.dni in self._socios:
            raise ValueError(f"Ya existe un socio con DNI {socio.dni}")
        self._socios[socio.dni] = socio

    def _buscar_libro(self, isbn):
        if isbn not in self._libros:
            raise ValueError(f"No existe un libro con ISBN {isbn}")
        return self._libros[isbn]

    def _buscar_socio(self, dni):
        if dni not in self._socios:
            raise ValueError(f"No existe un socio con DNI {dni}")
        return self._socios[dni]

    def prestar(self, isbn, dni):
        libro = self._buscar_libro(isbn)
        socio = self._buscar_socio(dni)
        # Validamos todo antes de cambiar algo, así un error no deja nada a medio hacer
        if not libro.disponible:
            raise ValueError(f"El libro '{libro.titulo}' no está disponible")
        if not socio.puede_pedir():
            raise ValueError(f"{socio.nombre} ya tiene el máximo de libros")
        libro.prestar()
        socio.agregar_libro(libro)

    def devolver(self, isbn, dni):
        libro = self._buscar_libro(isbn)
        socio = self._buscar_socio(dni)
        if libro not in socio.libros:
            raise ValueError(f"{socio.nombre} no tiene el libro '{libro.titulo}'")
        socio.quitar_libro(libro)
        libro.devolver()

    def libros_disponibles(self):
        return [libro for libro in self._libros.values() if libro.disponible]

In [6]:
# Prueben su biblioteca acá
bib = Biblioteca("Biblioteca POO")
bib.agregar_libro(Libro("Rayuela", "Cortázar", "978-1"))
bib.agregar_libro(Libro("Ficciones", "Borges", "978-2"))
bib.registrar_socio(Socio("Ana", "30111222"))

bib.prestar("978-1", "30111222")
for l in bib.libros_disponibles():
    print(l)            # solo Ficciones
bib.devolver("978-1", "30111222")
print(len(bib.libros_disponibles()))  # 2

[978-2] Ficciones - Borges (Disponible)
2


---
## Parte 4 — Pruebas automáticas (2 min)

Ejecuten la siguiente celda. **Si todo está bien, se imprime `¡Todas las pruebas pasaron!`**.
Si aparece un `AssertionError`, lean el mensaje: les dice qué parte falla.

In [7]:
def probar(condicion, mensaje):
    assert condicion, mensaje

def debe_fallar(funcion, mensaje):
    try:
        funcion()
    except ValueError:
        return
    probar(False, mensaje)

# --- Libro ---
l = Libro("T", "A", "I1")
probar(l.disponible, "Un libro nuevo debe estar disponible")
l.prestar()
probar(not l.disponible, "prestar() no marca el libro como prestado")
debe_fallar(l.prestar, "No se puede prestar un libro ya prestado")
l.devolver()
probar(l.disponible, "devolver() no marca el libro como disponible")
debe_fallar(l.devolver, "No se puede devolver un libro que no está prestado")
probar("I1" in str(l) and "T" in str(l), "__str__ debe mostrar ISBN y título")
try:
    l.disponible = False
    probar(False, "disponible debe ser una propiedad de solo lectura")
except AttributeError:
    pass

# --- Socio ---
s = Socio("Test", "D1")
probar(len(s.libros) == 0 and s.puede_pedir(), "Un socio nuevo no tiene libros y puede pedir")
for i in range(3):
    s.agregar_libro(Libro("T", "A", f"X{i}"))
probar(not s.puede_pedir(), "Con 3 libros el socio no puede pedir más")
debe_fallar(lambda: s.agregar_libro(Libro("T", "A", "X9")), "No se puede superar el máximo de 3 libros")
debe_fallar(lambda: s.quitar_libro(Libro("T", "A", "NO")), "No se puede quitar un libro que el socio no tiene")

# --- Biblioteca ---
b = Biblioteca("Test")
for i in range(5):
    b.agregar_libro(Libro("T", "A", f"B{i}"))
b.registrar_socio(Socio("S", "D1"))
b.registrar_socio(Socio("R", "D2"))
debe_fallar(lambda: b.agregar_libro(Libro("T", "A", "B0")), "No se permiten ISBN repetidos")
debe_fallar(lambda: b.registrar_socio(Socio("X", "D1")), "No se permiten DNI repetidos")
debe_fallar(lambda: b.prestar("NOEXISTE", "D1"), "prestar debe fallar si el libro no existe")
debe_fallar(lambda: b.prestar("B0", "NOEXISTE"), "prestar debe fallar si el socio no existe")

b.prestar("B0", "D1")
probar(len(b.libros_disponibles()) == 4, "prestar no saca el libro de los disponibles")
debe_fallar(lambda: b.prestar("B0", "D2"), "No se puede prestar un libro ya prestado")
debe_fallar(lambda: b.devolver("B0", "D2"), "Un socio no puede devolver un libro que no tiene")

b.prestar("B1", "D1")
b.prestar("B2", "D1")
debe_fallar(lambda: b.prestar("B3", "D1"), "Un socio con 3 libros no puede pedir otro")
probar(len(b.libros_disponibles()) == 2, "Si el préstamo falla, el libro debe seguir disponible")

b.devolver("B0", "D1")
probar(len(b.libros_disponibles()) == 3, "devolver no vuelve a poner el libro como disponible")
b.prestar("B3", "D1")  # ahora sí puede

print("¡Todas las pruebas pasaron!")

¡Todas las pruebas pasaron!


---
## Parte 5 — Subir el trabajo a GitHub (10 min)

Igual que en el TP del Banco: un integrante crea el repositorio **público** `tp-poo-biblioteca-grupoX`, agrega a los demás como colaboradores y van subiendo los cambios con:
```bash
git add .
git commit -m "Describe qué se agregó o cambió"
git push
```
> Si no pueden usar la terminal, también pueden subir los archivos desde la web: **Add file → Upload files** → **Commit changes**.

### Estructura de la entrega
**Cada clase en su propio archivo `.py`, y el programa principal también.** No se entrega el notebook.

```
tp-poo-biblioteca-grupoX/
├── README.md        ← nombres de los integrantes + cómo ejecutarlo
├── libro.py         ← class Libro
├── socio.py         ← class Socio
├── biblioteca.py    ← class Biblioteca
└── main.py          ← crea la biblioteca, hace préstamos y devoluciones, y corre las pruebas de la Parte 4
```

Al ejecutar `python main.py`, el programa tiene que mostrar algunos préstamos y devoluciones y terminar con **`¡Todas las pruebas pasaron!`**.
El repositorio tiene que tener **al menos 3 commits**.

### Entregar
Subir al EVA